## データの確認

In [1]:
import json

with open(
    "../dataset/fashionpedia_raw/annotations/instances_attributes_train2020.json"
) as f:
    data = json.load(f)

print(data.keys())

dict_keys(['annotations', 'images', 'info', 'licenses', 'categories', 'attributes'])


カテゴリの一覧確認

In [2]:
for cat in data["categories"]:
    print(cat["id"], cat["name"])

0 shirt, blouse
1 top, t-shirt, sweatshirt
2 sweater
3 cardigan
4 jacket
5 vest
6 pants
7 shorts
8 skirt
9 coat
10 dress
11 jumpsuit
12 cape
13 glasses
14 hat
15 headband, head covering, hair accessory
16 tie
17 glove
18 watch
19 belt
20 leg warmer
21 tights, stockings
22 sock
23 shoe
24 bag, wallet
25 scarf
26 umbrella
27 hood
28 collar
29 lapel
30 epaulette
31 sleeve
32 pocket
33 neckline
34 buckle
35 zipper
36 applique
37 bead
38 bow
39 flower
40 fringe
41 ribbon
42 rivet
43 ruffle
44 sequin
45 tassel


## 13クラスへ統合

In [6]:
#統合後のクラス
    # 0  outer
    # 1  tops
    # 2  bottoms
    # 3  dress
    # 4  shoes
    # 5  bag
    # 6  hat
    # 7  watch
    # 8  glasses
    # 9  belt
    # 10 neck_accessory
    # 11 leg_wear
    # 12 glove

CLASS_MAP = {
    # outerwear
    4: 0,   # jacket
    5: 0,   # vest
    9: 0,   # coat
    12: 0,  # cape

    # tops
    0: 1,   # shirt, blouse
    1: 1,   # top, t-shirt, sweatshirt
    2: 1,   # sweater
    3: 1,   # cardigan

    # bottoms
    6: 2,   # pants
    7: 2,   # shorts
    8: 2,   # skirt

    # dress
    10: 3,  # dress
    11: 3,  # jumpsuit

    # shoes
    23: 4,  # shoe

    # bag
    24: 5,  # bag, wallet

    # accessory
    # hat
    14: 6,  
    15: 6,  # headband
  
    # watch
    18: 7,     

    # glasses　
    13: 8,  

    # belt
    19: 9,  

    #neck_accesory
    16: 10,  # tie
    25: 10,  # scarf
    
    #leg_wear
    20: 11,  # leg warmer
    21: 11,  # tights
    22: 11,  # sock
    
    # glove
    17: 12
}

NEW_CATEGORIES = [
    {"id": 0, "name": "outer"},
    {"id": 1, "name": "tops"},
    {"id": 2, "name": "bottoms"},
    {"id": 3, "name": "dress"},
    {"id": 4, "name": "shoes"},
    {"id": 5, "name": "bag"},
    {"id": 6, "name": "hat"},
    {"id": 7, "name": "watch"},
    {"id": 8, "name": "glasses"},
    {"id": 9, "name": "belt"},
    {"id":10, "name": "neck_accesory"},
    {"id":11, "name": "leg_wear"},
    {"id":12, "name": "glove"}
]


変換実行用関数

In [7]:
def convert_fashionpedia(input_json, output_json):

    print(f"Loading {input_json}")

    with open(input_json, "r") as f:
        data = json.load(f)

    new_data = deepcopy(data)

    # ------------------------
    # annotation変換
    # ------------------------
    new_annotations = []

    for ann in data["annotations"]:

        old_class = ann["category_id"]

        # 不要クラス削除
        if old_class not in CLASS_MAP:
            continue

        ann = ann.copy()
        ann["category_id"] = CLASS_MAP[old_class]

        new_annotations.append(ann)

    # ------------------------
    # annotationが残った画像だけ残す
    # ------------------------
    valid_image_ids = set()

    for ann in new_annotations:
        valid_image_ids.add(ann["image_id"])

    new_images = []

    for img in data["images"]:
        if img["id"] in valid_image_ids:
            new_images.append(img)

    # ------------------------
    # 保存
    # ------------------------
    new_data["annotations"] = new_annotations
    new_data["images"] = new_images
    new_data["categories"] = NEW_CATEGORIES

    with open(output_json, "w") as f:
        json.dump(new_data, f)

    print()
    print("saved:", output_json)
    print("images:", len(new_images))
    print("annotations:", len(new_annotations))
    print("categories:", len(NEW_CATEGORIES))

変換実行

In [10]:
from copy import deepcopy
# ==========================
# train
# ==========================

convert_fashionpedia(
    "../dataset/fashionpedia_raw/annotations/instances_attributes_train2020.json",
    "../dataset/fashionpedia_13class/annotations/train13.json"
)

# ==========================
# val
# ==========================

convert_fashionpedia(
    "../dataset/fashionpedia_raw/annotations/instances_attributes_val2020.json",
    "../dataset/fashionpedia_13class/annotations/val13.json"
)

Loading ../dataset/fashionpedia_raw/annotations/instances_attributes_train2020.json

saved: ../dataset/fashionpedia_13class/annotations/train13.json
images: 45622
annotations: 162925
categories: 13
Loading ../dataset/fashionpedia_raw/annotations/instances_attributes_val2020.json

saved: ../dataset/fashionpedia_13class/annotations/val13.json
images: 1158
annotations: 4683
categories: 13


実行結果の確認

In [11]:
with open(
    "../dataset/fashionpedia_13class/annotations/train13.json"
) as f:
    train_data = json.load(f)

with open(
    "../dataset/fashionpedia_13class/annotations/val13.json"
) as f:
    val_data = json.load(f)
    
print(f"train_data:{train_data["categories"]}")
print(f"val_data:{val_data["categories"]}")

train_data:[{'id': 0, 'name': 'outer'}, {'id': 1, 'name': 'tops'}, {'id': 2, 'name': 'bottoms'}, {'id': 3, 'name': 'dress'}, {'id': 4, 'name': 'shoes'}, {'id': 5, 'name': 'bag'}, {'id': 6, 'name': 'hat'}, {'id': 7, 'name': 'watch'}, {'id': 8, 'name': 'glasses'}, {'id': 9, 'name': 'belt'}, {'id': 10, 'name': 'neck_accesory'}, {'id': 11, 'name': 'leg_wear'}, {'id': 12, 'name': 'glove'}]
val_data:[{'id': 0, 'name': 'outer'}, {'id': 1, 'name': 'tops'}, {'id': 2, 'name': 'bottoms'}, {'id': 3, 'name': 'dress'}, {'id': 4, 'name': 'shoes'}, {'id': 5, 'name': 'bag'}, {'id': 6, 'name': 'hat'}, {'id': 7, 'name': 'watch'}, {'id': 8, 'name': 'glasses'}, {'id': 9, 'name': 'belt'}, {'id': 10, 'name': 'neck_accesory'}, {'id': 11, 'name': 'leg_wear'}, {'id': 12, 'name': 'glove'}]


ラベルの偏りを確認

In [13]:
from collections import Counter

counter_train = Counter()
counter_val = Counter()

for ann in train_data["annotations"]:
    counter_train[ann["category_id"]] += 1

for ann in val_data["annotations"]:
    counter_val[ann["category_id"]] += 1
    
print("==== Train Class Distribution ====")

for cat in train_data["categories"]:
    cid = cat["id"]
    print(
        f"{cat['name']:10s} : {counter_train[cid]}"
    )
print("\n")   
print("==== Val Class Distribution ====")

for cat in val_data["categories"]:
    cid = cat["id"]
    print(f"{cat['name']:10s} : {counter_val[cid]}")

==== Train Class Distribution ====
outer      : 11828
tops       : 25310
bottoms    : 20216
dress      : 19661
shoes      : 46374
bag        : 7217
hat        : 5988
watch      : 3389
glasses    : 4855
belt       : 6851
neck_accesory : 2831
leg_wear   : 7020
glove      : 1385


==== Val Class Distribution ====
outer      : 314
tops       : 612
bottoms    : 582
dress      : 529
shoes      : 1566
bag        : 214
hat        : 183
watch      : 84
glasses    : 130
belt       : 164
neck_accesory : 51
leg_wear   : 223
glove      : 31


In [14]:
print("train_images:", len(train_data["images"]))
print("train_annotations:", len(train_data["annotations"]))
print("val_images:", len(val_data["images"]))
print("val_annotations:", len(val_data["annotations"]))

train_images: 45622
train_annotations: 162925
val_images: 1158
val_annotations: 4683


## COCO→YOLO変換

13クラスの中でyolo形式に変換できないサンプルを削除

In [15]:
def clean_coco_for_yolo(input_json, output_json):
    with open(input_json, "r") as f:
        data = json.load(f)

    clean_annotations = []
    removed = 0

    for ann in data["annotations"]:
        # bboxチェック
        bbox = ann.get("bbox")
        if (
            bbox is None
            or not isinstance(bbox, list)
            or len(bbox) != 4
            or any(v is None for v in bbox)
        ):
            removed += 1
            continue

        # segmentationチェック
        seg = ann.get("segmentation")
        if not isinstance(seg, list):
            removed += 1
            continue

        valid_polygons = []

        for poly in seg:
            if not isinstance(poly, list):
                continue

            if len(poly) < 6:
                continue

            if len(poly) % 2 != 0:
                continue

            if any(v is None for v in poly):
                continue

            valid_polygons.append(poly)

        if len(valid_polygons) == 0:
            removed += 1
            continue

        ann = ann.copy()
        ann["segmentation"] = valid_polygons
        clean_annotations.append(ann)

    valid_image_ids = {ann["image_id"] for ann in clean_annotations}
    clean_images = [
        img for img in data["images"]
        if img["id"] in valid_image_ids
    ]

    data["annotations"] = clean_annotations
    data["images"] = clean_images

    with open(output_json, "w") as f:
        json.dump(data, f)

    print("saved:", output_json)
    print("removed:", removed)
    print("images:", len(clean_images))
    print("annotations:", len(clean_annotations))

clean_coco_for_yolo(
    "../dataset/fashionpedia_13class/annotations/train13.json",
    "../dataset/fashionpedia_13class_clean/annotations/train13_clean.json"
)

clean_coco_for_yolo(
    "../dataset/fashionpedia_13class/annotations/val13.json",
    "../dataset/fashionpedia_13class_clean/annotations/val13_clean.json"
)

saved: ../dataset/fashionpedia_13class_clean/annotations/train13_clean.json
removed: 7183
images: 45604
annotations: 155742
saved: ../dataset/fashionpedia_13class_clean/annotations/val13_clean.json
removed: 226
images: 1158
annotations: 4457


coco→yolo変換の実行

In [17]:
from collections import defaultdict
from pathlib import Path

def coco_json_to_yolo_seg(json_path, output_label_dir):
    json_path = Path(json_path)
    output_label_dir = Path(output_label_dir)

    output_label_dir.mkdir(parents=True, exist_ok=True)

    with open(json_path, "r") as f:
        data = json.load(f)

    images = {
        img["id"]: img for img in data["images"]
    }

    labels_by_image = defaultdict(list)

    skipped = 0

    for ann in data["annotations"]:
        image_id = ann["image_id"]
        category_id = ann["category_id"]
        segs = ann["segmentation"]

        img = images[image_id]
        w = img["width"]
        h = img["height"]

        for poly in segs:
            if len(poly) < 6:
                skipped += 1
                continue

            if len(poly) % 2 != 0:
                skipped += 1
                continue

            normalized = []

            for i in range(0, len(poly), 2):
                x = poly[i] / w
                y = poly[i + 1] / h

                # 念のため0〜1にclip
                x = max(0.0, min(1.0, x))
                y = max(0.0, min(1.0, y))

                normalized.extend([x, y])

            line = str(category_id) + " " + " ".join(
                f"{v:.6f}" for v in normalized
            )

            labels_by_image[image_id].append(line)

    for image_id, lines in labels_by_image.items():
        file_name = images[image_id]["file_name"]
        txt_name = Path(file_name).stem + ".txt"

        with open(output_label_dir / txt_name, "w") as f:
            f.write("\n".join(lines))

    print("json:", json_path)
    print("labels:", len(labels_by_image))
    print("skipped polygons:", skipped)

coco_json_to_yolo_seg(
    "../dataset/fashionpedia_13class_clean/annotations/train13_clean.json",
    "../dataset/fashionpedia_yolo_13class/labels/train"
)

coco_json_to_yolo_seg(
    "../dataset/fashionpedia_13class_clean/annotations/val13_clean.json",
    "../dataset/fashionpedia_yolo_13class/labels/val"
)

json: ../dataset/fashionpedia_13class_clean/annotations/train13_clean.json
labels: 45604
skipped polygons: 0
json: ../dataset/fashionpedia_13class_clean/annotations/val13_clean.json
labels: 1158
skipped polygons: 0


確認

In [18]:
train_labels = list(Path("../dataset/fashionpedia_yolo_13class/labels/train").glob("*.txt"))
val_labels = list(Path("../dataset/fashionpedia_yolo_13class/labels/val").glob("*.txt"))

print("train labels:", len(train_labels))
print("val labels:", len(val_labels))

print(train_labels[:3])

train labels: 45604
val labels: 1158
[PosixPath('../dataset/fashionpedia_yolo_13class/labels/train/ad5634dfb6d4859aef38fd4bdd5f0380.txt'), PosixPath('../dataset/fashionpedia_yolo_13class/labels/train/426b9796a52977b22b425159494b0d7b.txt'), PosixPath('../dataset/fashionpedia_yolo_13class/labels/train/92d4b6482c10135dc944e9960fd8b849.txt')]


## yoloの学習用画像配置を準備

In [19]:
import json
import shutil
from pathlib import Path
from tqdm import tqdm

# JSON読み込み
with open("../dataset/fashionpedia_13class_clean/annotations/train13_clean.json") as f:
    train_json = json.load(f)

with open("../dataset/fashionpedia_13class_clean/annotations/val13_clean.json") as f:
    val_json = json.load(f)

# ファイル名取得
train_files = {
    img["file_name"]
    for img in train_json["images"]
}

val_files = {
    img["file_name"]
    for img in val_json["images"]
}

# 元画像
src_train = Path("../dataset/fashionpedia_raw/train")
src_val = Path("../dataset/fashionpedia_raw/test")

# 出力先
dst_train = Path("../dataset/fashionpedia_yolo_13class/images/train")
dst_val = Path("../dataset/fashionpedia_yolo_13class/images/val")

dst_train.mkdir(parents=True, exist_ok=True)
dst_val.mkdir(parents=True, exist_ok=True)

# trainコピー
for fname in tqdm(train_files):
    src = src_train / fname
    if src.exists():
        shutil.copy(src, dst_train / fname)

# valコピー
for fname in tqdm(val_files):
    src = src_val / fname
    if src.exists():
        shutil.copy(src, dst_val / fname)

print("done")

100%|███████████████████████████████████████| 1158/1158 [01:33<00:00, 12.43it/s]

done


fashion.yaml作成

In [ ]:
yaml_text = """
path: ../dataset/fashionpedia_yolo_13class

train: images/train
val: images/val

names:
  0: outer
  1: tops
  2: bottoms
  3: dress
  4: shoes
  5: bag
  6: hat
  7: watch
  8: glasses
  9: belt
  10: neck_accessory
  11: leg_wear
  12: glove
"""

with open("../dataset/fashionpedia_yolo_13class/fashion.yaml", "w") as f:
    f.write(yaml_text)

print("fashion.yaml created")

fashion.yaml created


確認

In [21]:
print(Path("../dataset/fashionpedia_yolo_13class/fashion.yaml").exists())

print(len(list(
    Path("../dataset/fashionpedia_yolo_13class/images/train").glob("*")
)))

print(len(list(
    Path("../dataset/fashionpedia_yolo_13class/images/val").glob("*")
)))

True
45604
1158
